# CIFAR-10 Grad-CAM Research Project


##  Setup & Dependencies


In [38]:
import os, math, random, time, logging, warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
import torchvision.models as tv_models
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.stats import spearmanr

In [39]:
for d in ['outputs/heatmaps', 'outputs/curves', 'outputs/sanity_checks', 'outputs/eval', 'models', 'data']:
    Path(d).mkdir(parents=True, exist_ok=True)

## Configuration


In [40]:
CFG = {
    'seed': 42,
    'data': {
        'dataset':     'cifar10',
        'root':        './data',
        'num_workers': 0,
        'mean': [0.4914, 0.4822, 0.4465],
        'std':  [0.2470, 0.2435, 0.2616],
    },
    'augmentation': {
        'random_crop_size':    32,
        'random_crop_padding': 4,
        'horizontal_flip_prob': 0.5,
    },
    'training': {
        'epochs':       100,
        'batch_size':   128,
        'lr':           0.01,
        'momentum':     0.9,
        'weight_decay': 5e-4,
    },
    'models': {
        'save_dir': './models',
        'baseline_cnn': {'dropout': 0.5},
    },
    'evaluation': {
        'library_parity_threshold': 0.95,
    },
    'paths': {
        'outputs':       './outputs',
        'heatmaps':      './outputs/heatmaps',
        'curves':        './outputs/curves',
        'sanity_checks': './outputs/sanity_checks',
        'eval':          './outputs/eval',
    },
}

## Utilities


In [41]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'[seed] All RNG sources fixed to {seed}')
def get_logger(name: str, log_file: str = None) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(name)s | %(message)s',
                            datefmt='%Y-%m-%d %H:%M:%S')
    if not logger.handlers:
        ch = logging.StreamHandler()
        ch.setFormatter(fmt)
        logger.addHandler(ch)
        if log_file:
            Path(log_file).parent.mkdir(parents=True, exist_ok=True)
            fh = logging.FileHandler(log_file)
            fh.setFormatter(fmt)
            logger.addHandler(fh)
    return logger
def save_checkpoint(state: dict, path: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)
    print(f'[checkpoint] Saved → {path}')
def load_checkpoint(path: str, model: nn.Module, optimizer=None, device='cpu'):
    state = torch.load(path, map_location=device)
    model.load_state_dict(state['model_state_dict'])
    if optimizer and 'optimizer_state_dict' in state:
        optimizer.load_state_dict(state['optimizer_state_dict'])
    print(f"[checkpoint] Loaded ← {path}  (epoch {state.get('epoch', '?')})")
    return state
def denormalize(tensor: torch.Tensor, mean: list, std: list) -> np.ndarray:
    m = torch.tensor(mean).view(3, 1, 1)
    s = torch.tensor(std).view(3, 1, 1)
    img = tensor.cpu().float() * s + m
    img = img.clamp(0, 1).permute(1, 2, 0).numpy()
    return (img * 255).astype(np.uint8)
def plot_training_curves(train_losses, val_losses, train_accs, val_accs,
                         save_path: str, title: str = 'Training Curves') -> None:
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    epochs = range(1, len(train_losses) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    axes[0].plot(epochs, train_losses, label='Train', color='#2563EB')
    axes[0].plot(epochs, val_losses,   label='Val',   color='#DC2626', linestyle='--')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(epochs, train_accs, label='Train', color='#2563EB')
    axes[1].plot(epochs, val_accs,   label='Val',   color='#DC2626', linestyle='--')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(save_path)

def get_device() -> torch.device:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[device] Using: {device}'
          + (f' ({torch.cuda.get_device_name(0)})' if device.type == 'cuda' else ''))
    return device
set_seed(CFG['seed'])
DEVICE = get_device()

[seed] All RNG sources fixed to 42
[device] Using: cuda (NVIDIA RTX 5000 Ada Generation)



## Data Pipeline


In [42]:
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']
def get_transforms(cfg: dict, split: str):
    mean = cfg['data']['mean']
    std  = cfg['data']['std']
    aug  = cfg['augmentation']
    normalize = T.Normalize(mean=mean, std=std)

    if split == 'train':
        return T.Compose([
            T.RandomCrop(aug['random_crop_size'], padding=aug['random_crop_padding']),
            T.RandomHorizontalFlip(p=aug['horizontal_flip_prob']),
            T.ToTensor(),
            normalize,
        ])
    else:
        return T.Compose([T.ToTensor(), normalize])

def get_dataloaders(cfg: dict):
    root = cfg['data']['root']
    bs   = cfg['training']['batch_size']
    nw   = cfg['data']['num_workers']

    train_set = torchvision.datasets.CIFAR10(
        root=root, train=True,  download=True, transform=get_transforms(cfg, 'train'))
    test_set  = torchvision.datasets.CIFAR10(
        root=root, train=False, download=True, transform=get_transforms(cfg, 'val'))

    train_loader = DataLoader(train_set, batch_size=bs, shuffle=True,
                              num_workers=nw, pin_memory=True)
    val_loader   = DataLoader(test_set,  batch_size=bs, shuffle=False,
                              num_workers=nw, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=bs, shuffle=False,
                              num_workers=nw, pin_memory=True)

    print(f'Train: {len(train_set):,} samples | Val/Test: {len(test_set):,} samples')
    return train_loader, val_loader, test_loader

def verify_class_balance(dataset, split_name: str) -> None:
    labels  = [dataset[i][1] for i in range(len(dataset))]
    counts  = np.bincount(labels, minlength=10)
    expected = 5000 if split_name == 'train' else 1000
    print(f'\n[{split_name}] Class counts:')
    for i, (cls, cnt) in enumerate(zip(CLASSES, counts)):
        print(f'  {i:2d}. {cls:<12s}: {cnt}')
    for i, cnt in enumerate(counts):
        assert cnt == expected, f'Class {CLASSES[i]} has {cnt} samples, expected {expected}.'
    print(f'[{split_name}]  Class balance verified ({expected} per class)')

def save_sample_grid(dataset, cfg: dict, save_path: str, n: int = 16) -> None:
    """Save grid of n augmented training images — visual label sanity check."""
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    mean = cfg['data']['mean']
    std  = cfg['data']['std']
    indices = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    fig.suptitle('Augmented training samples (sanity check)', fontsize=11)
    for ax, idx in zip(axes.flat, indices):
        img_tensor, label = dataset[int(idx)]
        ax.imshow(denormalize(img_tensor, mean, std))
        ax.set_title(CLASSES[label], fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'[sanity] Sample grid saved → {save_path}')

def run_data_sanity_checks(cfg: dict) -> None:
    root = cfg['data']['root']
    raw_train = torchvision.datasets.CIFAR10(root=root, train=True,  download=True, transform=T.ToTensor())
    raw_test  = torchvision.datasets.CIFAR10(root=root, train=False, download=True, transform=T.ToTensor())
    verify_class_balance(raw_train, 'train')
    verify_class_balance(raw_test,  'test')
    aug_train = torchvision.datasets.CIFAR10(
        root=root, train=True, download=True, transform=get_transforms(cfg, 'train'))
    save_sample_grid(aug_train, cfg,
                     save_path=cfg['paths']['sanity_checks'] + '/data_samples.png')

In [43]:
# Run data sanity checks
run_data_sanity_checks(CFG)

Files already downloaded and verified
Files already downloaded and verified

[train] Class counts:
   0. airplane    : 5000
   1. automobile  : 5000
   2. bird        : 5000
   3. cat         : 5000
   4. deer        : 5000
   5. dog         : 5000
   6. frog        : 5000
   7. horse       : 5000
   8. ship        : 5000
   9. truck       : 5000
[train]  Class balance verified (5000 per class)

[test] Class counts:
   0. airplane    : 1000
   1. automobile  : 1000
   2. bird        : 1000
   3. cat         : 1000
   4. deer        : 1000
   5. dog         : 1000
   6. frog        : 1000
   7. horse       : 1000
   8. ship        : 1000
   9. truck       : 1000
[test]  Class balance verified (1000 per class)
Files already downloaded and verified
[sanity] Sample grid saved → ./outputs/sanity_checks/data_samples.png


## Model Definitions


In [44]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes: int = 10, dropout: float = 0.5):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3,  32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2))  # 32→16

        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2))  # 16→8

        # Grad-CAM hooks on layer3
        self.layer3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2)) # 8→4

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.layer3(self.layer2(self.layer1(x))))

def build_resnet18(pretrained: bool = True, num_classes: int = 10) -> nn.Module:
    if pretrained:
        weights = tv_models.ResNet18_Weights.IMAGENET1K_V1
        model   = tv_models.resnet18(weights=weights)
        print('[model] ResNet18 loaded with ImageNet pretrained weights')
    else:
        model = tv_models.resnet18(weights=None)
        print('[model] ResNet18 initialized from scratch (random weights)')
    model.conv1  = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def build_model(model_name: str, cfg: dict) -> nn.Module:
    if model_name == 'baseline_cnn':
        return BaselineCNN(num_classes=10, dropout=cfg['models']['baseline_cnn']['dropout'])
    elif model_name == 'resnet18_pretrained':
        return build_resnet18(pretrained=True,  num_classes=10)
    elif model_name == 'resnet18_scratch':
        return build_resnet18(pretrained=False, num_classes=10)
    else:
        raise ValueError(f'Unknown model: {model_name}')
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
for name in ['baseline_cnn', 'resnet18_scratch', 'resnet18_pretrained']:
    m = build_model(name, CFG)
    print(f'{name}: {count_parameters(m):,} trainable parameters')

baseline_cnn: 620,810 trainable parameters
[model] ResNet18 initialized from scratch (random weights)
resnet18_scratch: 11,173,962 trainable parameters
[model] ResNet18 loaded with ImageNet pretrained weights
resnet18_pretrained: 11,173,962 trainable parameters


##  Training Loop



In [45]:
train_logger = get_logger('train', log_file='outputs/train.log')


def train_one_epoch(model, loader, criterion, optimizer, device, epoch):
    model.train()
    total_loss = total_correct = total_samples = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * images.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += images.size(0)
    return total_loss / total_samples, 100.0 * total_correct / total_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_samples = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss   = criterion(logits, labels)
        total_loss    += loss.item() * images.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += images.size(0)
    return total_loss / total_samples, 100.0 * total_correct / total_samples


def check_initial_loss(loss: float, num_classes: int = 10, tol: float = 0.3) -> None:
    expected  = math.log(num_classes)
    deviation = abs(loss - expected)
    status    = 'PASSED' if deviation <= tol else 'FAILED'
    train_logger.info(
        f'[sanity] Epoch-1 loss: got {loss:.4f}, expected ~{expected:.4f} '
        f'(±{tol}) | {status}'
    )
    if deviation > tol:
        train_logger.warning(
            '[sanity] Check: weight init, data pipeline, class balance, label encoding.')


def train(model_name: str, cfg: dict, epochs_override: int = None) -> nn.Module:
    set_seed(cfg['seed'])
    device = get_device()

    train_loader, val_loader, _ = get_dataloaders(cfg)
    model     = build_model(model_name, cfg).to(device)
    n_params  = count_parameters(model)
    train_logger.info(f'[model] {model_name} | {n_params:,} trainable params')
    optimizer = optim.SGD(
        model.parameters(),
        lr=cfg['training']['lr'],
        momentum=cfg['training']['momentum'],
        weight_decay=cfg['training']['weight_decay'],
    )
    num_epochs = epochs_override or cfg['training']['epochs']
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion  = nn.CrossEntropyLoss()

    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    best_val_acc = 0.0
    save_dir     = cfg['models']['save_dir']
    Path(save_dir).mkdir(parents=True, exist_ok=True)

    train_logger.info(f'Starting: {model_name} | {num_epochs} epochs')

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
        val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        if epoch == 1:
            check_initial_loss(train_loss)

        train_losses.append(train_loss);  val_losses.append(val_loss)
        train_accs.append(train_acc);     val_accs.append(val_acc)

        gen_gap = val_loss - train_loss
        train_logger.info(
            f'Epoch {epoch:3d}/{num_epochs} | '
            f'TrainLoss={train_loss:.4f} TrainAcc={train_acc:.1f}% | '
            f'ValLoss={val_loss:.4f} ValAcc={val_acc:.1f}% | '
            f'Gap={gen_gap:+.4f} | LR={scheduler.get_last_lr()[0]:.6f} | '
            f'{time.time()-t0:.1f}s'
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_checkpoint({
                'epoch': epoch, 'model_name': model_name,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc, 'val_loss': val_loss, 'train_loss': train_loss,
            }, path=f'{save_dir}/{model_name}_best.pth')
    save_checkpoint({
        'epoch': num_epochs, 'model_name': model_name,
        'model_state_dict': model.state_dict(),
        'train_losses': train_losses, 'val_losses': val_losses,
        'train_accs': train_accs,     'val_accs': val_accs,
    }, path=f'{save_dir}/{model_name}_final.pth')
    plot_training_curves(
        train_losses, val_losses, train_accs, val_accs,
        save_path=f"{cfg['paths']['curves']}/{model_name}_curves.png",
        title=f'{model_name} — Training Curves (best val: {best_val_acc:.1f}%)'
    )
    train_logger.info(f'Training complete. Best val accuracy: {best_val_acc:.2f}%')
    best_model = build_model(model_name, cfg).to(device)
    load_checkpoint(f'{save_dir}/{model_name}_best.pth', best_model, device=str(device))
    return best_model

In [46]:
TRAINED_MODELS = {}

for model_name in ['baseline_cnn', 'resnet18_scratch', 'resnet18_pretrained']:
    print('\n')
    print(f'Training: {model_name}')
    print('\n')
    TRAINED_MODELS[model_name] = train(model_name, CFG)



Training: baseline_cnn


[seed] All RNG sources fixed to 42
[device] Using: cuda (NVIDIA RTX 5000 Ada Generation)
Files already downloaded and verified
Files already downloaded and verified


2026-06-12 07:07:13 | INFO | train | [model] baseline_cnn | 620,810 trainable params
2026-06-12 07:07:13 | INFO | train | Starting: baseline_cnn | 100 epochs


Train: 50,000 samples | Val/Test: 10,000 samples


2026-06-12 07:07:34 | INFO | train | [sanity] Epoch-1 loss: got 1.5994, expected ~2.3026 (±0.3) | FAILED
2026-06-12 07:07:34 | WARNING | train | [sanity] Check: weight init, data pipeline, class balance, label encoding.
2026-06-12 07:07:34 | INFO | train | Epoch   1/100 | TrainLoss=1.5994 TrainAcc=40.7% | ValLoss=1.1797 ValAcc=56.6% | Gap=-0.4197 | LR=0.009998 | 21.7s


[checkpoint] Saved → ./models/baseline_cnn_best.pth


...

[checkpoint] Saved → ./models/resnet18_pretrained_best.pth


2026-06-12 09:04:47 | INFO | train | Epoch  79/100 | TrainLoss=0.0010 TrainAcc=100.0% | ValLoss=0.1555 ValAcc=96.1% | Gap=+0.1545 | LR=0.001049 | 26.3s
2026-06-12 09:05:13 | INFO | train | Epoch  80/100 | TrainLoss=0.0010 TrainAcc=100.0% | ValLoss=0.1547 ValAcc=96.0% | Gap=+0.1536 | LR=0.000955 | 26.1s
2026-06-12 09:05:40 | INFO | train | Epoch  81/100 | TrainLoss=0.0009 TrainAcc=100.0% | ValLoss=0.1537 ValAcc=96.0% | Gap=+0.1528 | LR=0.000865 | 26.9s
2026-06-12 09:06:06 | INFO | train | Epoch  82/100 | TrainLoss=0.0010 TrainAcc=100.0% | ValLoss=0.1551 ValAcc=96.1% | Gap=+0.1541 | LR=0.000778 | 26.4s
2026-06-12 09:06:33 | INFO | train | Epoch  83/100 | TrainLoss=0.0010 TrainAcc=100.0% | ValLoss=0.1535 ValAcc=96.1% | Gap=+0.1525 | LR=0.000696 | 26.9s
2026-06-12 09:07:00 | INFO | train | Epoch  84/100 | TrainLoss=0.0010 TrainAcc=100.0% | ValLoss=0.1551 ValAcc=96.2% | Gap=+0.1541 | LR=0.000618 | 26.8s
2026-06-12 09:07:26 | INFO | train | Epoch  85/100 | TrainLoss=0.0009 TrainAcc=100.0% | 

[checkpoint] Saved → ./models/resnet18_pretrained_final.pth


2026-06-12 09:14:11 | INFO | train | Training complete. Best val accuracy: 96.16%


./outputs/curves/resnet18_pretrained_curves.png
[model] ResNet18 loaded with ImageNet pretrained weights
[checkpoint] Loaded ← ./models/resnet18_pretrained_best.pth  (epoch 78)


In [47]:
torch.save(model.state_dict(), "baseline_cnn")
torch.save(model.state_dict(), "resnet18_scratch")
torch.save(model.state_dict(), "resnet18_pretrained")

##  Grad-CAM Implementation (from scratch)

In [48]:
class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model        = model
        self.target_layer = target_layer
        self._feature_maps = None
        self._gradients    = None
        self._register_hooks()

    def _register_hooks(self) -> None:
        def forward_hook(module, inp, out):
            self._feature_maps = out.detach()   # (B, C, H, W)
        def backward_hook(module, grad_in, grad_out):
            self._gradients = grad_out[0].detach()  # (B, C, H, W)
        self._fwd = self.target_layer.register_forward_hook(forward_hook)
        self._bwd = self.target_layer.register_full_backward_hook(backward_hook)

    def remove_hooks(self) -> None:
        self._fwd.remove()
        self._bwd.remove()

    def __call__(self, input_tensor: torch.Tensor,
                 target_class: int = None,
                 input_size: tuple = (32, 32)) -> np.ndarray:
        self.model.eval()
        self.model.zero_grad()

        logits = self.model(input_tensor)
        if target_class is None:
            target_class = logits.argmax(dim=1).item()
        score = logits[0, target_class]
        score.backward()
        weights = self._gradients.mean(dim=(2, 3))
        cam = (weights[:, :, None, None] * self._feature_maps).sum(dim=1).squeeze(0)
        cam = F.relu(cam)
        cam = F.interpolate(
            cam.unsqueeze(0).unsqueeze(0),
            size=input_size, mode='bilinear', align_corners=False
        ).squeeze().cpu().numpy()
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        else:
            cam = np.zeros_like(cam)
        return cam
    def __enter__(self): return self
    def __exit__(self, *args): self.remove_hooks()

def get_target_layer(model: nn.Module, model_name: str) -> nn.Module:
    if 'resnet18' in model_name:
        return model.layer4[-1]
    elif 'baseline_cnn' in model_name:
        return model.layer3
    else:
        raise ValueError(model_name)


def overlay_heatmap(image_np: np.ndarray, cam: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    heatmap = (cm.jet(cam)[:, :, :3] * 255).astype(np.uint8)
    blended = (1 - alpha) * image_np.astype(np.float32) / 255 + \
               alpha      * heatmap.astype(np.float32) / 255
    return (blended * 255).astype(np.uint8)


def save_heatmap_grid(model, model_name, dataset, cfg, save_path,
                      device=torch.device('cpu')) -> None:
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    mean = cfg['data']['mean'];  std = cfg['data']['std']
    target_layer = get_target_layer(model, model_name)
    class_images = {}
    for idx in range(len(dataset)):
        img_t, label = dataset[idx]
        if label not in class_images:
            class_images[label] = (img_t, label)
        if len(class_images) == 10:
            break
    fig, axes = plt.subplots(10, 2, figsize=(5, 25))
    fig.suptitle(f'Grad-CAM — {model_name}', fontsize=11, fontweight='bold')
    with GradCAM(model, target_layer) as gcam:
        for cls_idx in range(10):
            img_t, _ = class_images[cls_idx]
            img_np   = denormalize(img_t, mean, std)
            cam      = gcam(img_t.unsqueeze(0).to(device), target_class=cls_idx)
            overlay  = overlay_heatmap(img_np, cam)
            axes[cls_idx, 0].imshow(img_np)
            axes[cls_idx, 0].set_ylabel(CLASSES[cls_idx], fontsize=7,
                                        rotation=0, labelpad=40, va='center')
            axes[cls_idx, 0].axis('off')
            axes[cls_idx, 1].imshow(overlay)
            axes[cls_idx, 1].axis('off')

    axes[0, 0].set_title('Original', fontsize=9)
    axes[0, 1].set_title('Grad-CAM', fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'[gradcam] Heatmap grid saved → {save_path}')

In [49]:
test_dataset = torchvision.datasets.CIFAR10(
    root=CFG['data']['root'], train=False, download=True,
    transform=get_transforms(CFG, 'val')
)

for model_name, model in TRAINED_MODELS.items():
    model.eval()
    save_heatmap_grid(
        model, model_name, test_dataset, CFG,
        save_path=f"{CFG['paths']['heatmaps']}/{model_name}_gradcam_grid.png",
        device=DEVICE
    )

Files already downloaded and verified
[gradcam] Heatmap grid saved → ./outputs/heatmaps/baseline_cnn_gradcam_grid.png
[gradcam] Heatmap grid saved → ./outputs/heatmaps/resnet18_scratch_gradcam_grid.png
[gradcam] Heatmap grid saved → ./outputs/heatmaps/resnet18_pretrained_gradcam_grid.png


## Library Parity Validation


In [50]:
def verify_against_library(model, model_name: str, image_tensor: torch.Tensor,
                            target_class: int, cfg: dict,
                            threshold: float = 0.95,
                            device=torch.device('cpu')) -> float:
    try:
        from pytorch_grad_cam import GradCAM as LibGradCAM
        from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    except ImportError:
        print(' Run: pip install grad-cam')
        return float('nan')

    target_layer = get_target_layer(model, model_name)
    input_t      = image_tensor.unsqueeze(0).to(device)
    with GradCAM(model, target_layer) as gcam:
        our_cam = gcam(input_t, target_class=target_class)
    lib_gcam = LibGradCAM(model=model, target_layers=[target_layer])
    lib_cam  = lib_gcam(input_tensor=input_t,
                        targets=[ClassifierOutputTarget(target_class)])[0]

    r, p = spearmanr(our_cam.flatten(), lib_cam.flatten())
    status = 'PASSED' if r >= threshold else 'FAILED'
    print(f'[parity] {model_name}: Spearman r = {r:.4f} (p={p:.2e}) '
          f'| threshold = {threshold} | {status}')
    if r < threshold:
        print('averaging dims.')
    return r

parity_results = {}
sample_img, sample_label = test_dataset[0]
threshold = CFG['evaluation']['library_parity_threshold']

for model_name, model in TRAINED_MODELS.items():
    model.eval()
    r = verify_against_library(
        model, model_name, sample_img, sample_label, CFG,
        threshold=threshold, device=DEVICE
    )
    parity_results[model_name] = r

print('\nParity Results Summary:')
for name, r in parity_results.items():
    status = 'True' if (not np.isnan(r) and r >= threshold) else 'False'
    print(f'  {status} {name}: r = {r:.4f}')

[parity] baseline_cnn: Spearman r = 1.0000 (p=0.00e+00) | threshold = 0.95 | PASSED
[parity] resnet18_scratch: Spearman r = 1.0000 (p=0.00e+00) | threshold = 0.95 | PASSED
[parity] resnet18_pretrained: Spearman r = 1.0000 (p=0.00e+00) | threshold = 0.95 | PASSED

Parity Results Summary:
  True baseline_cnn: r = 1.0000
  True resnet18_scratch: r = 1.0000
  True resnet18_pretrained: r = 1.0000


## Evaluation — Accuracy & Confusion Matrices


In [51]:
def run_full_evaluation(trained_models: dict, cfg: dict, device) -> dict:
    _, val_loader, _ = get_dataloaders(cfg)
    criterion        = nn.CrossEntropyLoss()
    eval_path        = cfg['paths']['eval']
    Path(eval_path).mkdir(parents=True, exist_ok=True)

    results = {}

    for model_name, model in trained_models.items():
        model.eval()

        # Collect all predictions
        all_preds, all_labels = [], []
        total_loss = total_samples = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                logits  = model(images)
                loss    = criterion(logits, labels)
                total_loss    += loss.item() * images.size(0)
                total_samples += images.size(0)
                all_preds.extend(logits.argmax(1).cpu().tolist())
                all_labels.extend(labels.cpu().tolist())
        acc  = 100.0 * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
        loss = total_loss / total_samples
        results[model_name] = {'accuracy': acc, 'loss': loss}
        print(f'[eval] {model_name}: Acc={acc:.2f}%  Loss={loss:.4f}')
        cm_matrix = np.zeros((10, 10), dtype=int)
        for pred, true in zip(all_preds, all_labels):
            cm_matrix[true, pred] += 1
        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(cm_matrix, interpolation='nearest', cmap='Blues')
        plt.colorbar(im)
        ax.set(xticks=range(10), yticks=range(10),
               xticklabels=CLASSES, yticklabels=CLASSES,
               xlabel='Predicted', ylabel='True',
               title=f'{model_name} — Confusion Matrix (Acc={acc:.1f}%)')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        thresh = cm_matrix.max() / 2
        for i in range(10):
            for j in range(10):
                ax.text(j, i, cm_matrix[i, j], ha='center', va='center',
                        color='white' if cm_matrix[i, j] > thresh else 'black', fontsize=7)
        plt.tight_layout()
        save_p = f'{eval_path}/{model_name}_confusion.png'
        plt.savefig(save_p, dpi=150, bbox_inches='tight')
        plt.close()
        print(f'[eval] Confusion matrix saved → {save_p}')
    names = list(results.keys())
    accs  = [results[n]['accuracy'] for n in names]
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(names, accs, color=['#3B82F6', '#10B981', '#F59E0B'], width=0.5)
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title('Model Accuracy Comparison — CIFAR-10')
    ax.set_ylim(0, 105)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{eval_path}/accuracy_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[eval] Accuracy comparison saved → {eval_path}/accuracy_comparison.png")
    return results
eval_results = run_full_evaluation(TRAINED_MODELS, CFG, DEVICE)
print(f'{"Model":<30} {"Accuracy":>10} {"Loss":>10}')
for name, res in eval_results.items():
    print(f'{name:<25} {res["accuracy"]:>9.2f}% {res["loss"]:>10.4f}')

Files already downloaded and verified
Files already downloaded and verified
Train: 50,000 samples | Val/Test: 10,000 samples
[eval] baseline_cnn: Acc=87.10%  Loss=0.3817
[eval] Confusion matrix saved → ./outputs/eval/baseline_cnn_confusion.png
[eval] resnet18_scratch: Acc=92.93%  Loss=0.2829
[eval] Confusion matrix saved → ./outputs/eval/resnet18_scratch_confusion.png
[eval] resnet18_pretrained: Acc=96.16%  Loss=0.1550
[eval] Confusion matrix saved → ./outputs/eval/resnet18_pretrained_confusion.png
[eval] Accuracy comparison saved → ./outputs/eval/accuracy_comparison.png
Model                            Accuracy       Loss
baseline_cnn                  87.10%     0.3817
resnet18_scratch              92.93%     0.2829
resnet18_pretrained           96.16%     0.1550


## Adebayo Sanity Checks

In [52]:
def model_randomization_test(model: nn.Module, model_name: str, image_tensor: torch.Tensor,
                              cfg: dict, target_class: int,
                              save_path: str, device=torch.device('cpu')) -> None:
    import copy
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    mean = cfg['data']['mean'];  std = cfg['data']['std']
    input_t = image_tensor.unsqueeze(0).to(device)
    layers = [(name, mod) for name, mod in model.named_modules()
              if len(list(mod.parameters(recurse=False))) > 0]
    layers_to_randomize = layers[::-1]
    stages = min(5, len(layers_to_randomize))
    model_copy = copy.deepcopy(model)
    target_layer = get_target_layer(model_copy, model_name)
    fig, axes = plt.subplots(1, stages + 1, figsize=((stages + 1) * 3, 3))
    fig.suptitle(f'Model Randomization Test — {model_name}', fontsize=10)
    with GradCAM(model_copy, target_layer) as gcam:
        cam = gcam(input_t, target_class=target_class)
    img_np = denormalize(image_tensor, mean, std)
    axes[0].imshow(overlay_heatmap(img_np, cam))
    axes[0].set_title('Original', fontsize=8); axes[0].axis('off')
    step = max(1, len(layers_to_randomize) // stages)
    for i in range(1, stages + 1):
        for _, mod in layers_to_randomize[(i-1)*step : i*step]:
            for p in mod.parameters(recurse=False):
                nn.init.normal_(p)
        target_layer = get_target_layer(model_copy, model_name)
        with GradCAM(model_copy, target_layer) as gcam:
            cam = gcam(input_t, target_class=target_class)
        axes[i].imshow(overlay_heatmap(img_np, cam))
        axes[i].set_title(f'Rand stage {i}', fontsize=8)
        axes[i].axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'[sanity] {save_path}')
def data_randomization_test(model_name: str, cfg: dict,
                             test_dataset, target_class: int,
                             save_path: str, shuffle_epochs: int = 10,
                             device=torch.device('cpu')) -> None:
    from torch.utils.data import TensorDataset
    import copy

    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    mean = cfg['data']['mean'];  std = cfg['data']['std']
    print(f'[sanity] Training model on shuffled labels ({shuffle_epochs} epochs)...')
    root = cfg['data']['root']
    raw_train = torchvision.datasets.CIFAR10(
        root=root, train=True, download=True, transform=get_transforms(cfg, 'train'))
    shuffled_labels = torch.randperm(len(raw_train)) % 10
    class ShuffledDataset(torch.utils.data.Dataset):
        def __init__(self, base, new_labels):
            self.base = base
            self.labels = new_labels
        def __len__(self): return len(self.base)
        def __getitem__(self, idx):
            img, _ = self.base[idx]
            return img, self.labels[idx].item()
    shuffled_ds     = ShuffledDataset(raw_train, shuffled_labels)
    shuffled_loader = DataLoader(shuffled_ds, batch_size=cfg['training']['batch_size'],
                                 shuffle=True, num_workers=cfg['data']['num_workers'])
    rand_model = build_model(model_name, cfg).to(device)
    optimizer  = optim.SGD(rand_model.parameters(), lr=cfg['training']['lr'],
                           momentum=cfg['training']['momentum'],
                           weight_decay=cfg['training']['weight_decay'])
    criterion  = nn.CrossEntropyLoss()
    for ep in range(1, shuffle_epochs + 1):
        loss, acc = train_one_epoch(rand_model, shuffled_loader, criterion, optimizer, device, ep)
        print(f'  Shuffle epoch {ep}/{shuffle_epochs}: loss={loss:.4f} acc={acc:.1f}%')
    img_t, _    = test_dataset[0]
    img_np      = denormalize(img_t, mean, std)
    input_t     = img_t.unsqueeze(0).to(device)
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    fig.suptitle(f'Data Randomization Test — {model_name}', fontsize=10)
    orig_model  = TRAINED_MODELS[model_name]
    for ax, (mdl, lbl) in zip(axes, [(orig_model, 'Trained (correct labels)'),
                                      (rand_model, 'Trained (shuffled labels)')]):
        tl = get_target_layer(mdl, model_name)
        with GradCAM(mdl, tl) as gcam:
            cam = gcam(input_t, target_class=target_class)
        ax.imshow(overlay_heatmap(img_np, cam))
        ax.set_title(lbl, fontsize=7)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'[sanity] Data randomization comparison saved → {save_path}')
sanity_path = CFG['paths']['sanity_checks']
target_class = 0
sample_img_t, _ = test_dataset[0]
for model_name, model in TRAINED_MODELS.items():
    model.eval()
    model_randomization_test(
        model, model_name, sample_img_t, CFG, target_class,
        save_path=f'{sanity_path}/{model_name}_model_rand.png',
        device=DEVICE
    )
    data_randomization_test(
        model_name, CFG, test_dataset, target_class,
        save_path=f'{sanity_path}/{model_name}_data_rand.png',
        shuffle_epochs=10,  # increase for more thorough training on shuffled data
        device=DEVICE
    )

[sanity] ./outputs/sanity_checks/baseline_cnn_model_rand.png
[sanity] Training model on shuffled labels (10 epochs)...
Files already downloaded and verified
  Shuffle epoch 1/10: loss=2.3074 acc=9.8%
  Shuffle epoch 2/10: loss=2.3029 acc=9.9%
  Shuffle epoch 3/10: loss=2.3029 acc=10.1%
  Shuffle epoch 4/10: loss=2.3029 acc=9.9%
  Shuffle epoch 5/10: loss=2.3028 acc=10.0%
  Shuffle epoch 6/10: loss=2.3027 acc=9.9%
  Shuffle epoch 7/10: loss=2.3029 acc=10.0%
  Shuffle epoch 8/10: loss=2.3028 acc=9.8%
  Shuffle epoch 9/10: loss=2.3028 acc=9.8%
  Shuffle epoch 10/10: loss=2.3028 acc=9.7%
[sanity] Data randomization comparison saved → ./outputs/sanity_checks/baseline_cnn_data_rand.png
[sanity] ./outputs/sanity_checks/resnet18_scratch_model_rand.png
[sanity] Training model on shuffled labels (10 epochs)...
Files already downloaded and verified
[model] ResNet18 initialized from scratch (random weights)
  Shuffle epoch 1/10: loss=2.3763 acc=10.0%
  Shuffle epoch 2/10: loss=2.3345 acc=10.1%
  S

## Summary & Results


In [55]:
print('\n CIFAR-10 Grad-CAM Project — Final Results Summary')
print('\n[ Accuracy Table ]')
print(f'{"Model":<25} {"Test Acc":>10} {"CE Loss":>10}')
for name, res in eval_results.items():
    print(f'{name:<25} {res["accuracy"]:>9.2f}% {res["loss"]:>10.4f}')
print('\n[ Library Parity (Spearman r) ]')
threshold = CFG['evaluation']['library_parity_threshold']
print(f'{"Model":<30} {"r":>3} {"Pass?":>17}')
for name, r in parity_results.items():
    passed = 'True' if (not np.isnan(r) and r >= threshold) else 'False'
    print(f'{name:<30} {r:>10f} {passed:>10}')
print('\n[ Output Files ]')
for pattern in [
    'outputs/curves/*_curves.png',
    'outputs/eval/*',
    'outputs/heatmaps/*',
    'outputs/sanity_checks/*',
    'models/*_best.pth',
]:
    import glob
    files = sorted(glob.glob(pattern))
    for f in files:
        print(f'  {f}')


 CIFAR-10 Grad-CAM Project — Final Results Summary

[ Accuracy Table ]
Model                       Test Acc    CE Loss
baseline_cnn                  87.10%     0.3817
resnet18_scratch              92.93%     0.2829
resnet18_pretrained           96.16%     0.1550

[ Library Parity (Spearman r) ]
Model                            r             Pass?
baseline_cnn                     1.000000       True
resnet18_scratch                 1.000000       True
resnet18_pretrained              1.000000       True

[ Output Files ]
  outputs/curves\baseline_cnn_curves.png
  outputs/curves\resnet18_pretrained_curves.png
  outputs/curves\resnet18_scratch_curves.png
  outputs/eval\accuracy_comparison.png
  outputs/eval\baseline_cnn_confusion.png
  outputs/eval\resnet18_pretrained_confusion.png
  outputs/eval\resnet18_scratch_confusion.png
  outputs/heatmaps\baseline_cnn_gradcam_grid.png
  outputs/heatmaps\resnet18_pretrained_gradcam_grid.png
  outputs/heatmaps\resnet18_scratch_gradcam_grid.png
  out